# Analyse des accidents de la route — Insights clés (BAAC 2024)

## Objectif
Présenter les principaux enseignements issus de l’analyse des accidents corporels de la circulation
en France à partir des données BAAC 2024, dans une logique d’aide à la décision et de prévention.

In [1]:
import sys
from pathlib import Path

# Ajoute la racine du projet au PYTHONPATH
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: C:\Users\Hannae\Documents\Projets\accidents-france-baac


In [2]:
import pandas as pd
from pathlib import Path
from IPython.display import Image, display


DATA = Path("../data/processed")
acc = pd.read_parquet(DATA / "accidents-2024.parquet")
usag = pd.read_parquet(DATA / "usagers-2024.parquet")

print(acc.shape, usag.shape)

(70248, 35) (125187, 3)


## Insight 1 — Concentration des accidents aux heures de pointe

In [3]:
from src.utils import save_fig
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))

acc_per_hour = acc["hour"].value_counts().sort_index()

acc_per_hour.plot(kind="bar")
plt.title("Accidents par heure")
plt.xlabel("Heure")
plt.ylabel("Nombre d'accidents")

path = save_fig("01_accidents_par_heure.png")
display(Image(filename=str(path)))

KeyboardInterrupt: 

Les accidents sont majoritairement concentrés aux heures de pointe,
notamment le matin et en fin de journée. Cette répartition suggère un lien fort avec les déplacements domicile–travail.


## Insight 2 — Répartition hebdomadaire des accidents

In [ ]:
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

plt.figure(figsize=(10, 5))

acc["weekday"].value_counts().reindex(order).plot(kind="bar")

plt.title("Nombre d'accidents par jour de la semaine")
plt.xlabel("Jour")
plt.ylabel("Nombre d'accidents")

path = save_fig("02_accidents_par_jour_semaine.png")
display(Image(filename=str(path)))

Les accidents sont plus fréquents en semaine, avec une baisse notable le dimanche, confirmant l’impact des déplacements professionnels sur la sinistralité routière.

## Insight 3 — Gravité des accidents impliquant les usagers

In [ ]:
plt.figure(figsize=(10, 5))

usag["grav_label"].value_counts().reindex(
    ["Indemne", "Blessé léger", "Blessé hospitalisé", "Tué"]
).plot(kind="bar")

plt.title("Gravité des accidents — usagers")
plt.xlabel("Gravité")
plt.ylabel("Nombre d'usagers")

path = save_fig("03_gravite_usagers.png")
display(Image(filename=str(path)))

La majorité des usagers impliqués dans les accidents sont indemnes ou légèrement blessés.
Les accidents mortels restent minoritaires, mais représentent un enjeu majeur en termes de prévention.

## Insight 4 — Périodes horaires à risque élevé

In [ ]:
# Jointure usagers + heure de l'accident
usag_acc = usag.merge(
    acc[["Num_Acc", "hour"]],
    on="Num_Acc",
    how="left"
)

plt.figure(figsize=(10, 5))

usag_acc.loc[usag_acc["grav_label"] == "Tué", "hour"] \
    .value_counts().sort_index() \
    .plot(kind="bar")

plt.title("Accidents mortels par heure")
plt.xlabel("Heure")
plt.ylabel("Nombre de tués")

path = save_fig("04_accidents_mortels_par_heure.png")
display(Image(filename=str(path)))

Le croisement entre l’heure de l’accident et la gravité met en évidence des périodes
à risque élevé, durant lesquelles les accidents mortels sont plus fréquents.
Ces créneaux pourraient constituer des cibles prioritaires pour des actions de prévention renforcées.

## Insight 5 — Top Départements

In [ ]:
[c for c in acc.columns if c.lower() in ["dep", "departement", "dep_acc"]]

plt.figure(figsize=(10, 5))

acc["dep"].value_counts().head(15).plot(kind="bar")

plt.title("Top 15 départements – accidents")
plt.xlabel("Département")
plt.ylabel("Nombre d'accidents")

path = save_fig("05_top_15_departements_accidents.png")
display(Image(filename=str(path)))

Les départements les plus peuplés concentrent logiquement le plus grand nombre d’accidents. Cette analyse en volume brut met en évidence des zones prioritaires en termes d’exposition, mais nécessite une normalisation par la population ou le trafic pour évaluer la dangerosité relative.